# Xử lý ngôn ngữ tự nhiên - CS221.Q21.KHTN
## Demo chương 7: Large Language Model (Language Modeling / Text Generation)

### Nhóm 10:
- Nguyễn Thành Sơn
- Đoàn Hồng Bảo
- Bùi Huỳnh Tây
- Nguyễn Quốc Phú

In [ ]:
!pip install -q transformers torch scikit-learn pyvi

In [ ]:
import os
import urllib.request
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. Cấu hình & Siêu tham số
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {DEVICE}")

# Sử dụng mô hình GPT-2 thay vì PhoBERT cho tác vụ sinh văn bản tiếng Anh
MODEL_NAME = "gpt2"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 5e-5

Running on device: cuda


In [ ]:
# ==========================================
# 2. Tải & Tiền xử lý dữ liệu Tiny Shakespeare
# ==========================================
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
filepath = "input.txt"

if not os.path.exists(filepath):
    print("Đang tải dữ liệu Tiny Shakespeare...")
    urllib.request.urlretrieve(url, filepath)

with open(filepath, 'r', encoding='utf-8') as f:
    text_data = f.read()

# Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

class ShakespeareDataset(Dataset):
    def __init__(self, text, tokenizer, max_len):
        self.tokens = tokenizer.encode(text, add_special_tokens=True)
        self.max_len = max_len
        self.num_samples = len(self.tokens) // max_len

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start_idx = idx * self.max_len
        end_idx = start_idx + self.max_len
        chunk = self.tokens[start_idx:end_idx]

        input_ids = torch.tensor(chunk, dtype=torch.long)

        return {
            'input_ids': input_ids,
            'attention_mask': torch.ones_like(input_ids),
            'labels': input_ids.clone()
        }

print("\nĐang chia tập dữ liệu...")
split_idx = int(len(text_data) * 0.8)
train_text = text_data[:split_idx]
test_text = text_data[split_idx:]

train_dataset = ShakespeareDataset(train_text, tokenizer, MAX_LEN)
test_dataset = ShakespeareDataset(test_text, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Tổng số batch huấn luyện: {len(train_loader)}")
print(f"Tổng số batch kiểm thử: {len(test_loader)}")


Đang chia tập dữ liệu...


Token indices sequence length is longer than the specified maximum sequence length for this model (267688 > 1024). Running this sequence through the model will result in indexing errors


Tổng số batch huấn luyện: 131
Tổng số batch kiểm thử: 35


In [ ]:
# ==========================================
# 3. Huấn luyện Mô hình (Fine-tuning GPT-2)
# ==========================================
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print(f"\nBắt đầu huấn luyện trên {DEVICE}...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        # Chú ý truyền đúng tham số bằng TỪ KHÓA (keyword arguments)
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()

        # Mô hình tự tính CrossEntropyLoss khi được truyền tham số labels
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if batch_idx % 100 == 0 and batch_idx > 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"-> Hoàn thành Epoch {epoch+1}/{EPOCHS} | Train Loss trung bình: {avg_loss:.4f}\n")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Bắt đầu huấn luyện trên cuda...
Epoch 1 | Batch 100/131 | Loss: 3.6029
-> Hoàn thành Epoch 1/3 | Train Loss trung bình: 3.8094

Epoch 2 | Batch 100/131 | Loss: 3.7468
-> Hoàn thành Epoch 2/3 | Train Loss trung bình: 3.5068

Epoch 3 | Batch 100/131 | Loss: 3.3218
-> Hoàn thành Epoch 3/3 | Train Loss trung bình: 3.3797



In [ ]:
# ==========================================
# 4. Đánh giá Mô hình trên tập Test
# ==========================================
import math
print("\nĐang đánh giá trên tập Test...")
model.eval()
total_eval_loss = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        # Tránh lỗi get_seq_length bằng cách truyền rõ tên biến
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_eval_loss += loss.item()

avg_eval_loss = total_eval_loss / len(test_loader)
# Perplexity là e mũ của Loss, đo lường độ bối rối của mô hình khi gặp văn bản mới (càng thấp càng tốt)
perplexity = math.exp(avg_eval_loss)

print(f"{'='*50}")
print(f"{'KẾT QUẢ ĐÁNH GIÁ (EVALUATION)':^50}")
print(f"{'='*50}")
print(f"Test Loss trung bình: {avg_eval_loss:.4f}")
print(f"Perplexity (Độ trôi chảy): {perplexity:.4f}")
print(f"{'='*50}")


Đang đánh giá trên tập Test...
          KẾT QUẢ ĐÁNH GIÁ (EVALUATION)           
Test Loss trung bình: 3.6720
Perplexity (Độ trôi chảy): 39.3286


In [ ]:
# ==========================================
# 5. Demo: Text Generation
# ==========================================
print("Đang sinh văn bản thử nghiệm...")
model.eval()

prompt = "ROMEO:\nO, speak again, bright angel!"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
attention_mask = torch.ones(input_ids.shape, dtype=torch.long, device=DEVICE)

with torch.no_grad():
    output_sequences = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=100,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)

print("\n" + "="*50)
print("VĂN BẢN MÔ HÌNH SINH RA")
print("="*50)
print(generated_text)
print("="*50)

Đang sinh văn bản thử nghiệm...

VĂN BẢN MÔ HÌNH SINH RA
ROMEO:
O, speak again, bright angel!--for I love thee as the sun; and thou art a woman. Give me thy handkerchief to cut off my neck of shame. Why wilt not that kiss which broke your heart? When didst Thou think it best when thine eyes were upon him in scorn? Yet is he dead yet so near his sepulchre's light? How long have we slept thus coldly since our last sleepings together!--that
